# All Trees

Decision trees, random forest, XGBoost

# Decision Trees for Classification

In [8]:
import os
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sdk1810/playtennis")

print("Path to dataset files:", path)
print("Files in directory:")
print(os.listdir(path))

Path to dataset files: /Users/ethanhersch/.cache/kagglehub/datasets/sdk1810/playtennis/versions/1
Files in directory:
['PlayTennis.csv']


In [9]:
import pandas as pd
import os

file_path = os.path.join(path, "PlayTennis.csv")

df = pd.read_csv(file_path)

df.head()

,outlook,temp,humidity,windy,play
0,sunny,hot,high,False,no
1,sunny,hot,high,True,no
2,overcast,hot,high,False,yes
3,rainy,mild,high,False,yes
4,rainy,cool,normal,False,yes


In [10]:
import numpy as np
import pandas as pd


class DecisionTreeNode:
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value  # used for leaf nodes


class DecisionTreeClassifierScratch:
    def __init__(self, max_depth=10, min_samples_split=2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self, X, y):
        X = np.array(X)
        y = np.array(y)
        self.root = self._build_tree(X, y, depth=0)

    def predict(self, X):
        X = np.array(X)
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _entropy(self, y):
        classes, counts = np.unique(y, return_counts=True)
        probs = counts / len(y)
        return -np.sum(probs * np.log2(probs + 1e-9))

    def _most_common_label(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]

    def _best_split(self, X, y):
        n_samples, n_features = X.shape
        best_gain = -1
        best_feature = None
        best_threshold = None

        parent_entropy = self._entropy(y)

        for feature in range(n_features):
            thresholds = np.unique(X[:, feature])

            for threshold in thresholds:
                left_mask = X[:, feature] <= threshold
                right_mask = X[:, feature] > threshold

                if np.sum(left_mask) == 0 or np.sum(right_mask) == 0:
                    continue

                y_left = y[left_mask]
                y_right = y[right_mask]

                n_left, n_right = len(y_left), len(y_right)
                child_entropy = (
                    (n_left / n_samples) * self._entropy(y_left)
                    + (n_right / n_samples) * self._entropy(y_right)
                )

                info_gain = parent_entropy - child_entropy

                if info_gain > best_gain:
                    best_gain = info_gain
                    best_feature = feature
                    best_threshold = threshold

        return best_feature, best_threshold, best_gain

    def _build_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # stopping conditions
        if (
            depth >= self.max_depth
            or n_labels == 1
            or n_samples < self.min_samples_split
        ):
            leaf_value = self._most_common_label(y)
            return DecisionTreeNode(value=leaf_value)

        best_feature, best_threshold, best_gain = self._best_split(X, y)

        if best_gain <= 0 or best_feature is None:
            leaf_value = self._most_common_label(y)
            return DecisionTreeNode(value=leaf_value)

        left_mask = X[:, best_feature] <= best_threshold
        right_mask = X[:, best_feature] > best_threshold

        left_subtree = self._build_tree(X[left_mask], y[left_mask], depth + 1)
        right_subtree = self._build_tree(X[right_mask], y[right_mask], depth + 1)

        return DecisionTreeNode(
            feature=best_feature,
            threshold=best_threshold,
            left=left_subtree,
            right=right_subtree,
        )

    def _traverse_tree(self, x, node):
        if node.value is not None:
            return node.value

        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)


In [15]:
# -------------------------
# Example on Tennis
# -------------------------
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

tree = DecisionTreeClassifierScratch(max_depth=5)
tree.fit(X_train, y_train)

preds = tree.predict(X_test)
print('Predictions:', preds)
print('Accuracy:', accuracy_score(y_test, preds))

Predictions: [1 0 2 1 1 0 1 2 1 1 2 0 0 0 0 1 2 1 1 2 0 2 0 2 2 2 2 2 0 0]
Accuracy: 1.0


# Random Forest

We can now use the custom Decision Tree Implementation to ensemble them into a forest. A **random forest** uses bagging of data in order to train the forest.

In [ ]:
class RandomForestClassifierScratch:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.trees = []

    def fit(self, X, y):
        self.trees = []
        for _ in range(self.n_estimators):
            tree = DecisionTreeClassifierScratch(
                max_depth=self.max_depth, min_samples_split=self.min_samples_split
            )
            n_samples = X.shape[0]
            indices = np.random.choice(n_samples, n_samples, replace=True)
            X_bootstrap = X[indices]
            y_bootstrap = y[indices]
            
            tree.fit(X_bootstrap, y_bootstrap)
            self.trees.append(tree)

    def predict(self, X):
        tree_preds = np.array([tree.predict(X) for tree in self.trees])
        tree_preds = np.swapaxes(tree_preds, 0, 1)
        y_pred = [np.bincount(pred).argmax() for pred in tree_preds]
        return np.array(y_pred)

In [ ]:
# presuming data is loaded in the decision tree example above

rf = RandomForestClassifierScratch(
    n_estimators=15, max_depth=5, min_samples_split=2
)
rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print(f" Random Forest Predictions: {y_pred}")
print(f"Random Forest Accuracy on Iris Dataset: {accuracy * 100:.2f}%")

# XGBoost

Now implementing XGBoost (since this uses a different implementation and is rarely implemented from scratch, we will be using the library).

However, XGBoost works as follows:
1. **Additive Training**. Instead of training all models at once, XGBoost uses an additive strategy. Let $\hat{y}_i^{(t)}$ be the prediction for the $i$-th instance at the $t$-th iteration. We add a new tree $f_t$ to improve the prediction:$$\hat{y}_i^{(t)} = \hat{y}_i^{(t-1)} + f_t(x_i)$$The goal at step $t$ is to find the tree $f_t$ that minimizes the overall objective function.

2. **The Objective Function**. The objective function in XGBoost consists of two parts: a loss function $L$ (which measures how well the model fits the training data) and a regularization term $\Omega$ (which penalizes model complexity to prevent overfitting).$$Obj(\theta) = \sum_{i=1}^{n} L(y_i, \hat{y}_i) + \sum_{k=1}^{K} \Omega(f_k)$$For a specific iteration $t$, the objective becomes:$$Obj^{(t)} = \sum_{i=1}^{n} L(y_i, \hat{y}_i^{(t-1)} + f_t(x_i)) + \Omega(f_t)$$

3. **Second-Order Taylor Approximation**. Standard gradient boosting optimization relies only on the first derivative (the gradient) of the loss function. XGBoost's major innovation is using a second-order Taylor expansion to approximate the objective function, which allows it to converge faster and find better splits.$$Obj^{(t)} \approx \sum_{i=1}^{n} \left[ L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$Where:$g_i$ is the gradient (first-order derivative) of the loss function.$h_i$ is the hessian (second-order derivative) of the loss function.Since $L(y_i, \hat{y}_i^{(t-1)})$ is a constant from the previous iteration, we can remove it to simplify the objective we need to minimize:$$Obj^{(t)} \approx \sum_{i=1}^{n} \left[ g_i f_t(x_i) + \frac{1}{2} h_i f_t^2(x_i) \right] + \Omega(f_t)$$

4. **Regularization Formulation**. XGBoost explicitly defines the complexity of a tree $\Omega(f)$. If a tree has $T$ leaves and the weight (output value) of each leaf $j$ is $w_j$, the regularization is defined as:$$\Omega(f) = \gamma T + \frac{1}{2} \lambda \sum_{j=1}^{T} w_j^2$$ $\gamma$ (Gamma) penalizes the number of leaves, controlling tree depth.$\lambda$ (Lambda) is L2 regularization on the leaf weights, preventing any single leaf from having an outsized influence.

5. **Calculating Leaf Weights and Split Gain**. By substituting the regularization term back into the objective and grouping the instances by their assigned leaf node, XGBoost calculates the optimal weight $w_j^*$ for a given leaf:$$w_j^* = -\frac{\sum_{i \in I_j} g_i}{\sum_{i \in I_j} h_i + \lambda}$$
To determine how to split a node, XGBoost evaluates the reduction in the objective function (the Gain) after the split. Let $G_L = \sum_{i \in I_L} g_i$ and $H_L = \sum_{i \in I_L} h_i$ represent the sum of gradients and hessians for the left child, and $G_R$, $H_R$ for the right child. The gain from a split is:$$Gain = \frac{1}{2} \left[ \frac{G_L^2}{H_L + \lambda} + \frac{G_R^2}{H_R + \lambda} - \frac{(G_L + G_R)^2}{H_L + H_R + \lambda} \right] - \gamma$$

In [ ]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(
    n_estimators=100,
    max_depth=5,
    random_state=42,
    n_jobs=-1,
)

xgb_clf.fit(X_train, y_train)
xgb_preds = xgb_clf.predict(X_test)
print("XGBoost predictions:", xgb_preds)
print("Accuracy:", accuracy_score(y_test, xgb_preds))